In [14]:
!pip install -q duckdb
!pip install -q polars

In [15]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import polars as pl

In [7]:
import duckdb

db_path = r"C:\Users\cincy\OneDrive\Documents\Sam R Work\nflmonad\db\nfl_analytics.duckdb"

con = duckdb.connect(
    database=db_path,
    read_only=True
)

In [8]:

con.sql("SHOW ALL TABLES").df()

,database,schema,name,column_names,column_types,temporary
0,nfl_analytics,main,int_game_base,"[season, week, game_id, season_type, game_date...","[INTEGER, INTEGER, VARCHAR, VARCHAR, DATE, VAR...",False
1,nfl_analytics,main,int_injury_team_impact,"[season, week, team, total_injury_burden, qb_o...","[INTEGER, INTEGER, VARCHAR, DOUBLE, DOUBLE, DO...",False
2,nfl_analytics,main,int_player_form,"[player_id, season, week, full_name, position,...","[VARCHAR, INTEGER, INTEGER, VARCHAR, VARCHAR, ...",False
3,nfl_analytics,main,int_player_game,"[player_id, season, week, full_name, position,...","[VARCHAR, INTEGER, INTEGER, VARCHAR, VARCHAR, ...",False
4,nfl_analytics,main,int_qb_team_context,"[player_id, season, week, game_id, full_name, ...","[VARCHAR, INTEGER, INTEGER, VARCHAR, VARCHAR, ...",False
5,nfl_analytics,main,int_team_form,"[season, week, team, game_id, game_num_in_seas...","[INTEGER, INTEGER, VARCHAR, VARCHAR, DOUBLE, D...",False
6,nfl_analytics,main,int_team_game,"[season, week, game_id, team, opponent, home_f...","[INTEGER, INTEGER, VARCHAR, VARCHAR, VARCHAR, ...",False
7,nfl_analytics,main,raw_depth_charts,"[season, week, club_code, game_type, depth_tea...","[INTEGER, INTEGER, VARCHAR, VARCHAR, VARCHAR, ...",False
8,nfl_analytics,main,raw_ff_opportunity,"[player_id, season, week, posteam, game_id, fu...","[VARCHAR, INTEGER, INTEGER, VARCHAR, VARCHAR, ...",False
9,nfl_analytics,main,raw_ftn_charting,"[season, week, ftn_game_id, nflverse_game_id, ...","[INTEGER, INTEGER, INTEGER, VARCHAR, INTEGER, ...",False


In [12]:
weekly_player_stats = con.sql("SELECT * FROM nfl_analytics.stg_player_week").df()
print(weekly_player_stats.columns)
weekly_player_stats.head()

Index(['player_id', 'season', 'week', 'full_name', 'position',
       'position_group', 'team', 'opponent', 'season_type', 'completions',
       'attempts', 'passing_yards', 'passing_tds', 'interceptions', 'sacks',
       'sack_yards', 'sack_fumbles', 'sack_fumbles_lost', 'passing_air_yards',
       'passing_yac', 'passing_epa', 'dakota', 'carries', 'rushing_yards',
       'rushing_tds', 'rushing_fumbles', 'rushing_fumbles_lost',
       'rushing_first_downs', 'rushing_epa', 'targets', 'receptions',
       'receiving_yards', 'receiving_tds', 'receiving_fumbles',
       'receiving_fumbles_lost', 'receiving_air_yards', 'receiving_yac',
       'receiving_first_downs', 'receiving_epa', 'target_share',
       'air_yards_share', 'wopr', 'racr', 'pacr', 'fantasy_points_std',
       'fantasy_points_ppr', 'ingestion_ts'],
      dtype='object')


,player_id,season,week,full_name,position,position_group,team,opponent,season_type,completions,...,receiving_first_downs,receiving_epa,target_share,air_yards_share,wopr,racr,pacr,fantasy_points_std,fantasy_points_ppr,ingestion_ts
0,0,1999,1,None,None,None,ATL,MIN,REG,0,...,0,0.000000,0.000000,NaN,NaN,0.0,0.0,0.0,0.0,2026-04-09 22:30:47.373869-04:00
1,00-0000003,1999,1,Abdul-Karim al-Jabbar,RB,RB,MIA,DEN,REG,0,...,0,0.292378,0.045455,NaN,NaN,0.0,0.0,12.7,13.7,2026-04-09 22:30:47.373869-04:00
2,00-0000007,1999,1,Rabih Abdullah,RB,RB,TB,NYG,REG,0,...,0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,2026-04-09 22:30:47.373869-04:00
3,00-0000008,1999,1,Rahim Abdullah,OLB,LB,CLE,PIT,REG,0,...,0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,2026-04-09 22:30:47.373869-04:00
4,00-0000017,1999,1,Donnie Abraham,CB,DB,TB,NYG,REG,0,...,0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,2026-04-09 22:30:47.373869-04:00


In [ ]:
player_df = con.sql("SELECT * FROM nfl_analytics.stg_players").df()

player_df.head()

,player_id,full_name,first_name,last_name,position,position_group,birth_date,college,height,weight,entry_year,rookie_year,draft_club,draft_number,active_flag,headshot_url,short_name,ingestion_ts
0,00-0028830,Isaako Aaitui,Isaako,Aaitui,NT,DL,1987-01-25,UNLV,76,307,<NA>,2011,None,<NA>,0,https://static.www.nfl.com/image/private/f_aut...,None,2026-04-09 22:31:57.860500-04:00
1,00-0038389,Israel Abanikanda,Israel,Abanikanda,RB,RB,2002-10-05,Pittsburgh,70,216,2023,2023,NYJ,143,1,https://static.www.nfl.com/image/upload/f_auto...,I.Abanikanda,2026-04-09 22:31:57.860500-04:00
2,00-0024644,Jon Abbate,Jon,Abbate,LB,OTHER,1985-06-18,Wake Forest,71,245,<NA>,2007,None,<NA>,0,https://static.www.nfl.com/image/private/f_aut...,None,2026-04-09 22:31:57.860500-04:00
3,ABB498348,Vince Abbott,Vincent,Abbott,K,K,1958-05-31,California State-Fullerton; Washington,71,207,<NA>,1987,None,<NA>,1,https://static.www.nfl.com/image/private/f_aut...,None,2026-04-09 22:31:57.860500-04:00
4,00-0031021,Jared Abbrederis,Jared,Abbrederis,WR,WR,1990-12-17,Wisconsin,73,195,2014,2014,GB,176,0,https://static.www.nfl.com/image/private/f_aut...,J.Abbrederis,2026-04-09 22:31:57.860500-04:00
